# Example experiment on the microscope-agnostic layer

This notebook walks the general flow a smart-microscopy workflow follows on top
of the layer. The workflow only ever talks to the layer - it never imports a
vendor driver, so the same notebook runs against any microscope that has an
adapter.

Every layer call is *send/receive*: it forwards your intent and the session
context to the driver and returns what the driver gives back. The layer never
moves a stage or computes anything itself - the driver does the work.

The flow:

1. **connect** - select the driver and open the session
2. **select states** - capture the reactivatable configuration for each scan mode
3. **get positions** - the positions to visit
4. **acquire** - set a mode, then for each position: set_xyz, acquire, save

## Setup

Register the bundled **mock driver** so the flow runs with no hardware. This is
scaffolding for the example only - in a real notebook the vendor driver is
already registered and you would just call `connect(...)`. Run this notebook with
its own folder as the working directory.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()  # .../microscopes/microscope_agnostic_layer
sys.path.insert(0, str(here.parents[0]))  # microscopes/ source root
sys.path.insert(0, str(here / "tests"))  # the mock driver

import mock_driver  # noqa: E402

mock_driver.register_mock()

## 1. Connect

`connect` does two things at once.

**Selects the driver** from `vendor` / `microscope` / `api-type` - the three
coordinates that locate a driver in the `drivers/vendor/microscope/api` tree.

**Sets up the coordinate system** from the reference `objective` and
`stage_type`. "Absolute" coordinates only mean something against a frame: the
objective fixes the optical reference (offsets between objectives are the
driver's job) and `stage_type` fixes the actuator frame that `set_xyz` works in.
(Auth - `client` / `password` - would also go here if the driver needs it.)

It then opens the session and discovers the capabilities. Omit any of these and
the vendor's defaults fill in.

In [ ]:
from microscope_agnostic_layer import connect

mic = connect(
    vendor="mock",  # driver location: vendor / microscope / api-type
    microscope="mock-scope",
    api="mock-api",
    objective="10x",  # coordinate frame: reference objective + stage
    stage_type="motoric",
)

The **capabilities** are the menu the driver advertises: for each selectable
axis, the available `options` and the currently `active` one. This is the single
source of truth - you read an option here and pass the same value back as an
argument (a stage for `set_xyz`, a `format` for `save`, ...). Omit the argument
and the layer uses `active`.

In [ ]:
mic.capabilities

## 2. Select states

A **state** is a snapshot of the instrument you can reactivate later. It has an
`immutable` part (an instrument/config fingerprint the driver uses to reject a
state captured elsewhere) and a `mutable` part (the settings that actually get
reapplied). The dict is opaque to the layer - only the driver interprets it.

We capture two modes, one cell each: a gentle **prescan** and a higher-power
**target** scan.

In [ ]:
# snapshot the current state, then lower the laser for a gentle overview scan
prescan = mic.get_state()
prescan["mutable"]["laser_power"] = 2.0
prescan

In [ ]:
# a second state: more power and gain for the detailed target scan
target = mic.get_state()
target["mutable"]["laser_power"] = 8.0
target["mutable"]["gain"] = 2.0
target

## 3. Get the positions

The positions to visit, captured at connect and returned in the canonical
(motoric) coordinate frame. A real driver would derive these from the holder
layout or a prescan analysis.

In [ ]:
positions = mic.get_initial_positions()
positions

## 4. Acquisition

Activate a mode with `set_state`, then run every position. In the loop:

- **set_xyz** moves to the target in the motoric frame. The `stages` argument
  chooses which actuator realizes each axis - here Z uses the **piezo** for fine
  focus, while X and Y stay on their active (motoric) stage. The coordinate is
  the same whichever actuator moves you there; the legal stage names come from
  `capabilities`.
- **acquire** with `backlash_correction=True` settles the stage via the right
  approach *before* the capture, so the image is taken at the true position.
- **save** writes the result; `format` and `procedure` default to the `active`
  options discovered at connect.

Re-run with `mic.set_state(target)` to acquire the target scan instead.

In [ ]:
mic.set_state(prescan)

In [ ]:
for i, pos in enumerate(positions):
    # move in the motoric frame; realize Z with the piezo for fine focus
    mic.set_xyz(pos["x"], pos["y"], pos["z"], stages={"z": "piezo"})

    # settle correctly before the capture
    mic.acquire(backlash_correction=True)

    # save with the active format/procedure
    print(mic.save(name=f"prescan_{i:03d}", position=pos))

## Done

Close the session. The mock has nothing to release, but a real driver may need
the teardown.

In [ ]:
mic.disconnect()